In [1]:

import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

df = pd.DataFrame({
    "annual_income": np.random.randint(200000, 2000000, n),
    "loan_amount": np.random.randint(50000, 1500000, n),
    "loan_term_months": np.random.choice(
        [12, 24, 36, 60, 120, 240], n
    ),
    "credit_history": np.random.choice([0, 1], n, p=[0.25, 0.75]),
    "employment_type": np.random.choice(
        ["Salaried", "Self-employed"], n
    ),
    "existing_monthly_debt": np.random.randint(0, 40000, n)
})

# Generate demonstration labels using a predefined rule.
# These are synthetic labels, not real bank decisions.
estimated_monthly_payment = (
    df["loan_amount"] / df["loan_term_months"]
)

monthly_income = df["annual_income"] / 12

df["loan_status"] = np.where(
    (df["credit_history"] == 1) &
    (estimated_monthly_payment + df["existing_monthly_debt"]
     < monthly_income * 0.40),
    "Eligible",
    "Not Eligible"
)

print(df.head())
print("\nDataset shape:", df.shape)
print("\nTarget distribution:")
print(df["loan_status"].value_counts())


   annual_income  loan_amount  loan_term_months  credit_history  \
0         321958      1024304                24               1   
1         871155      1403038                24               1   
2         331932       300875                60               1   
3        1614414       513389                12               1   
4         459178       120390                36               0   

  employment_type  existing_monthly_debt   loan_status  
0        Salaried                  36484  Not Eligible  
1        Salaried                  36182  Not Eligible  
2   Self-employed                    570      Eligible  
3        Salaried                  32254  Not Eligible  
4        Salaried                  19288  Not Eligible  

Dataset shape: (1000, 7)

Target distribution:
loan_status
Not Eligible    658
Eligible        342
Name: count, dtype: int64


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

X = df.drop(columns=["loan_status"])
y = df["loan_status"]

numeric_features = [
    "annual_income",
    "loan_amount",
    "loan_term_months",
    "credit_history",
    "existing_monthly_debt"
]

categorical_features = ["employment_type"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model.fit(X_train, y_train)

print("Model training completed successfully!")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))


Model training completed successfully!
Training records: 800
Testing records: 200


In [3]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

y_pred = model.predict(X_test)

print("Accuracy:", round(
    accuracy_score(y_test, y_pred) * 100, 2
), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test, y_pred,
    labels=["Not Eligible", "Eligible"]
))


Accuracy: 91.5 %

Classification Report:
              precision    recall  f1-score   support

    Eligible       0.87      0.88      0.88        68
Not Eligible       0.94      0.93      0.94       132

    accuracy                           0.92       200
   macro avg       0.90      0.91      0.91       200
weighted avg       0.92      0.92      0.92       200


Confusion Matrix:
[[123   9]
 [  8  60]]


In [4]:

import joblib

joblib.dump(model, "loan_eligibility_model.pkl")

print("Model saved successfully!")


Model saved successfully!


In [5]:

from google.colab import files

files.download("loan_eligibility_model.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>